# Semana 12: RAG (Retrieval-Augmented Generation) y Bases de Datos Vectoriales
## Notebook de Ejercicios (NB2) – Chatbot sobre Documentos Propios

**Propósito:** Crear un chatbot que responda preguntas basándose en documentos propios (política de empresa simulada) utilizando RAG.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar un pipeline completo de RAG: chunking, embeddings, FAISS, recuperación y generación.
- Probar el chatbot con preguntas reales.
- Evaluar cualitativamente las respuestas.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install sentence-transformers faiss-cpu transformers pypdf2

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
warnings.filterwarnings('ignore')

# Sentence Transformers para embeddings
from sentence_transformers import SentenceTransformer

# FAISS para búsqueda vectorial
import faiss

# Transformers para generación
from transformers import pipeline, GPT2Tokenizer, GPT2LMHeadModel

# Para manejo de PDFs (opcional)
import PyPDF2
from io import BytesIO
import requests

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Documentos Propios (Política de Empresa Inventada)

Creamos un conjunto de documentos que simulan la política interna de una empresa ficticia llamada **TechCorp**.

In [ ]:
# Documentos sobre política de empresa (simulados)
raw_documents = [
    """Política de Vacaciones: Los empleados de TechCorp tienen derecho a 22 días hábiles de vacaciones al año. 
    Las vacaciones deben ser solicitadas con al menos 2 semanas de antelación y aprobadas por el supervisor directo. 
    Los días no utilizados pueden acumularse hasta un máximo de 5 días al año siguiente.""",
    
    """Horario Laboral: La jornada laboral estándar es de 9:00 a 18:00, de lunes a viernes, con una hora de almuerzo. 
    TechCorp ofrece horario flexible, permitiendo a los empleados comenzar entre 8:00 y 10:00, siempre que completen 
    sus 8 horas diarias. Existe también la opción de teletrabajo hasta 2 días por semana, sujeto a aprobación.""",
    
    """Beneficios: TechCorp ofrece a sus empleados: seguro médico privado (cobertura nacional e internacional), 
    seguro dental, plan de pensiones con aportación de la empresa del 5% del salario, y vales de comida por valor de 11€ diarios. 
    Además, hay un programa de formación continua con un presupuesto anual de 2000€ por empleado.""",
    
    """Política de Permisos: Los empleados tienen derecho a permisos retribuidos en casos de: matrimonio (15 días), 
    nacimiento de hijo (5 días), fallecimiento de familiar directo (3 días), mudanza (2 días), y consultas médicas (tiempo necesario). 
    Los permisos deben justificarse documentalmente.""",
    
    """Evaluación del Desempeño: TechCorp realiza evaluaciones de desempeño semestrales (junio y diciembre). 
    El proceso incluye autoevaluación, evaluación por parte del supervisor y revisión de objetivos. 
    Los resultados determinan aumentos salariales y bonificaciones anuales, que pueden ser de hasta el 10% del salario base.""",
    
    """Código de Conducta: Se espera que los empleados mantengan un comportamiento profesional y ético. 
    Prohibido el acoso en cualquier forma, discriminación y uso indebido de recursos de la empresa. 
    Las violaciones pueden resultar en acciones disciplinarias, incluyendo el despido.""",
    
    """Política de Gastos: Los gastos de viaje y representación deben ser pre-aprobados y presentados dentro de los 30 días. 
    Se reembolsan gastos de alojamiento (hasta 150€/noche), dietas (50€/día) y transporte público. 
    Los recibos deben adjuntarse en la solicitud de reembolso.""",
    
    """Desarrollo Profesional: TechCorp fomenta el desarrollo profesional mediante programas de mentoría, 
    acceso a plataformas de aprendizaje online (Coursera, Udemy) y organización de workshops internos. 
    Los empleados pueden solicitar apoyo para certificaciones profesionales relevantes."""
]

print(f"Total de documentos: {len(raw_documents)}")
print("\n=== EJEMPLO DE DOCUMENTO ===")
print(raw_documents[0])

---
## 2. Chunking (División en Fragmentos)

Dividimos los documentos largos en fragmentos más pequeños para mejorar la recuperación.

In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):
    """
    Divide un texto en fragmentos de aproximadamente chunk_size palabras,
    con solapamiento para mantener contexto.
    """
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    
    return chunks

# Aplicar chunking a todos los documentos
all_chunks = []
for doc in raw_documents:
    chunks = chunk_text(doc, chunk_size=50, overlap=10)
    all_chunks.extend(chunks)

print(f"Número de fragmentos después de chunking: {len(all_chunks)}")
print("\n=== EJEMPLO DE FRAGMENTO ===")
print(all_chunks[0])

---
## 3. Generación de Embeddings con Sentence Transformers

Creamos vectores para cada fragmento de texto.

In [ ]:
# Cargar modelo de embeddings
print("Cargando modelo de Sentence Transformers...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings para todos los fragmentos
chunk_embeddings = embedding_model.encode(all_chunks)

print(f"Forma de los embeddings: {chunk_embeddings.shape}")
print(f"(num_fragmentos, dimension_embedding)")

---
## 4. Construcción del Índice FAISS

Almacenamos los embeddings en un índice para búsqueda eficiente.

In [ ]:
# Obtener dimensión de los embeddings
dimension = chunk_embeddings.shape[1]

# Crear índice FAISS
index = faiss.IndexFlatL2(dimension)

# Añadir embeddings al índice
index.add(chunk_embeddings.astype('float32'))

print(f"Número de vectores en el índice: {index.ntotal}")

# Guardar índice en disco
faiss.write_index(index, "techcorp_index.faiss")
print("Índice guardado como 'techcorp_index.faiss'")

---
## 5. Función de Recuperación de Fragmentos Relevantes

Dada una pregunta, buscamos los fragmentos más similares.

In [ ]:
def retrieve_relevant_chunks(query, embedding_model, index, chunks, k=3):
    """
    Recupera los k fragmentos más relevantes para la consulta.
    """
    # Generar embedding de la consulta
    query_embedding = embedding_model.encode([query]).astype('float32')
    
    # Buscar en el índice
    distances, indices = index.search(query_embedding, k)
    
    # Obtener fragmentos relevantes
    relevant_chunks = [chunks[idx] for idx in indices[0]]
    
    print(f"\nConsulta: {query}")
    print(f"Top {k} fragmentos relevantes:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"\n{i+1}. [Distancia: {dist:.4f}]")
        print(f"   {chunks[idx][:100]}...")
    
    return relevant_chunks, distances[0], indices[0]

# Probar recuperación
query_test = "¿Cuántos días de vacaciones tengo?"
chunks_rel, dists, idxs = retrieve_relevant_chunks(query_test, embedding_model, index, all_chunks)

---
## 6. Generación de Respuestas con GPT-2

Cargamos un modelo generativo para responder basado en los fragmentos recuperados.

In [ ]:
# Cargar modelo generativo (GPT-2)
print("Cargando GPT-2 para generación...")
generator = pipeline('text-generation', model='gpt2', device=0 if torch.cuda.is_available() else -1)

def generate_response(query, relevant_chunks, generator, max_length=150):
    """
    Genera una respuesta usando los fragmentos recuperados como contexto.
    """
    # Construir contexto con los fragmentos relevantes
    context = " ".join(relevant_chunks)
    
    # Construir prompt
    prompt = f"""Contexto: {context}

Pregunta: {query}

Respuesta basada en el contexto:"""
    
    # Generar respuesta
    response = generator(prompt, max_length=max_length, num_return_sequences=1, temperature=0.7)[0]['generated_text']
    
    # Extraer solo la respuesta (después del prompt)
    answer = response[len(prompt):].strip()
    
    return answer

# Probar generación
query_test = "¿Cuántos días de vacaciones tengo?"
relevant_chunks, _, _ = retrieve_relevant_chunks(query_test, embedding_model, index, all_chunks, k=2)
answer = generate_response(query_test, relevant_chunks, generator)

print(f"\n=== RESPUESTA GENERADA ===")
print(answer)

---
## 7. Chatbot Completo

Integramos recuperación y generación en un chatbot interactivo.

In [ ]:
def chatbot(query, embedding_model, index, chunks, generator, k=3):
    """
    Chatbot RAG completo.
    """
    # 1. Recuperar fragmentos relevantes
    relevant_chunks, distances, indices = retrieve_relevant_chunks(query, embedding_model, index, chunks, k)
    
    # 2. Generar respuesta
    answer = generate_response(query, relevant_chunks, generator)
    
    # 3. Mostrar resultado
    print(f"\n{'='*60}")
    print(f"🤖 CHATBOT RESPONDE:")
    print(f"{answer}")
    
    return answer, relevant_chunks

# Probar el chatbot con algunas preguntas
preguntas = [
    "¿Cuántos días de vacaciones tengo?",
    "¿Cuál es el horario de trabajo?",
    "¿Qué beneficios ofrece la empresa?",
    "¿Cómo son las evaluaciones de desempeño?",
    "¿Puedo trabajar desde casa?"
]

for pregunta in preguntas:
    print(f"\n{'#'*60}")
    print(f"🗣️ USUARIO: {pregunta}")
    respuesta, _ = chatbot(pregunta, embedding_model, index, all_chunks, generator, k=2)
    print(f"\n{'#'*60}")

---
## 8. Evaluación Cualitativa de Respuestas

Evaluamos manualmente la calidad de las respuestas del chatbot.

In [ ]:
def evaluate_response_quality(query, answer, relevant_chunks):
    """
    Evaluación cualitativa de la respuesta.
    """
    print(f"\n{'='*60}")
    print(f"EVALUACIÓN DE RESPUESTA")
    print(f"Pregunta: {query}")
    print(f"\nRespuesta generada:")
    print(f"{answer}")
    print(f"\nFragmentos utilizados:")
    for i, chunk in enumerate(relevant_chunks):
        print(f"  {i+1}. {chunk[:150]}...")
    
    print("\nCriterios de evaluación (1-5):")
    relevancia = input("¿La respuesta responde a la pregunta? (1-5): ")
    fidelidad = input("¿La respuesta se basa en el contexto? (1-5): ")
    coherencia = input("¿La respuesta es coherente y bien redactada? (1-5): ")
    
    return {
        'relevancia': int(relevancia),
        'fidelidad': int(fidelidad),
        'coherencia': int(coherencia),
        'promedio': (int(relevancia) + int(fidelidad) + int(coherencia)) / 3
    }

# Evaluar una respuesta específica
query_eval = "¿Qué beneficios ofrece la empresa?"
answer, chunks_eval = chatbot(query_eval, embedding_model, index, all_chunks, generator, k=2)
# Descomentar para evaluación interactiva
# scores = evaluate_response_quality(query_eval, answer, chunks_eval)
# print(f"\nPuntaje promedio: {scores['promedio']:.2f}/5")

---
## 9. (Opcional) Carga de Documentos desde PDF

Si tienes documentos PDF, puedes cargarlos y procesarlos.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Extrae texto de un archivo PDF.
    """
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text()
    return text

# Ejemplo de uso (descomentar si tienes un PDF)
# pdf_text = extract_text_from_pdf('mi_documento.pdf')
# print(pdf_text[:500])

---
## 10. Ejercicio para el Estudiante

1. **Amplía la base de documentos**: Añade más políticas (ej. política de teletrabajo, política de igualdad).

2. **Mejora el chunking**: Experimenta con diferentes tamaños de chunk y solapamiento.

3. **Cambia el modelo generativo**: Prueba con `'gpt2-medium'` o `'gpt2-large'` para respuestas más detalladas.

4. **Implementa un umbral de similitud**: Solo usa fragmentos con distancia menor a un umbral.

5. **Evalúa con preguntas ambiguas**: ¿Qué pasa si la pregunta no tiene respuesta en los documentos?

In [ ]:
# Espacio para el estudiante
# ...

---
## 11. Conclusiones

En este notebook hemos construido un chatbot RAG completo sobre documentos propios:

✔️ **Documentos**: Simulamos una política de empresa.

✔️ **Chunking**: Dividimos documentos en fragmentos manejables.

✔️ **Embeddings**: Generamos vectores semánticos con Sentence Transformers.

✔️ **Índice FAISS**: Almacenamos y buscamos fragmentos eficientemente.

✔️ **Recuperación**: Encontramos fragmentos relevantes para cada pregunta.

✔️ **Generación**: Usamos GPT-2 para generar respuestas contextualizadas.

✔️ **Evaluación**: Evaluamos cualitativamente las respuestas.

**Lección clave**: RAG permite crear chatbots personalizados que responden preguntas basándose en documentos específicos, combinando lo mejor de la recuperación de información y la generación de lenguaje.

---
**Fin del Notebook de Ejercicios - Semana 12 (NLP)**